#  **Smart recommendation of research papers**

- 0. Introduction
  - a. Theory of RAG
  - b. Extraction data from xml to dataset
- 1. RAG implementation
  - a. LangChain
  - b. RAG from scratch

## 0. Introduction

My goal for this module is to implement a RAG-based recommendation system. I will start by using LangChain, than by a custom implementation from scratch.

### a. Theory of RAG

### b. Extraction data from xml to dataset

a. Dataset

In [ ]:
# Frameworks
import os
from lxml import etree
import pandas as pd
from data_analysis_helper import DataAnalysisHelper
import numpy as np
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import DataFrameLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
all_top_tags = {"article", "inproceedings", "proceedings", "book", 
                "incollection", "phdthesis", "mastersthesis", "www", "person", "data"}

In [19]:
lis_range = list(range(1990, 1994))
new_lis_range = []
for i, idx in enumerate(lis_range):
    new_lis_range.append(str(lis_range[i]))

xml_path = r"C:\Users\wikto\OneDrive\Dokumenty\all-datasets\dblp-dataset\dblp.xml"

helper_class = DataAnalysisHelper(xml_path, all_top_tags)
publications = helper_class.parse_publications(new_lis_range, target_limit=100, threshold =False)

flatted_data = []

for year, records in publications.items():
    for record in records:
        flatted_data.append({
            "Year": year,
            "Type": record["type"],      
            "Author": record["author"],  
            "Title": record["title"],    
            "Journal": record["journal"],
            "Booktitle": record["booktitle"],
            "Address": record["address"],
            "Month": record["month"],
            "Publisher": record["publisher"],
            "Note": record["note"],
            "Publnr": record["publnr"],
            "Rel": record["rel"]
        })


df = pd.DataFrame(flatted_data)

df.head()



[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\wikto\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\wikto\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\wikto\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Limit reached for 1992....
Limit reached for 1993....
Limit reached for 1990....
Limit reached for 1991....
Limit reached for every year. Ending....


,Year,Type,Author,Title,Journal,Booktitle,Address,Month,Publisher,Note,Publnr,Rel
0,1990,book,Edward Cohen,Programming in the 1990s - An Introduction to ...,None,None,None,None,Springer,None,None,None
1,1990,book,David C. Luckham,Programming with Specifications - An Introduct...,None,None,None,None,Springer,None,None,None
2,1990,book,Melvin Fitting,First-Order Logic and Automated Theorem Proving,None,None,None,None,Springer,None,None,None
3,1990,book,Helmuth Partsch,Specification and Transformation of Programs -...,None,None,None,None,Springer,None,None,None
4,1990,book,"Hartmut Ehrig, Bernd Mahr",Fundamentals of Algebraic Specification 2,None,None,None,None,Springer,None,None,None


In [20]:
# Replace None to NaN for .isna()
df = df.replace('None', np.nan)

print(f"How many records in df: {len(df)}")

# NaN stats when threshold = False
print(f"How many NaN's in that df:")
df.isna().sum()

How many records in df: 400
How many NaN's in that df:


C:\Users\wikto\AppData\Local\Temp\ipykernel_27860\1879774033.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace('None', np.nan)


Year           0
Type           0
Author        56
Title          0
Journal      400
Booktitle    385
Address      400
Month        400
Publisher      8
Note         400
Publnr       400
Rel          400
dtype: int64

In [24]:
df = df[["Year", "Title", "Author", "Type"]]
df = df.dropna(subset=['Title'])
df

,Year,Title,Author,Type
0,1990,Programming in the 1990s - An Introduction to ...,Edward Cohen,book
1,1990,Programming with Specifications - An Introduct...,David C. Luckham,book
2,1990,First-Order Logic and Automated Theorem Proving,Melvin Fitting,book
3,1990,Specification and Transformation of Programs -...,Helmuth Partsch,book
4,1990,Fundamentals of Algebraic Specification 2,"Hartmut Ehrig, Bernd Mahr",book
...,...,...,...,...
395,1993,Datenbank-Integration von Ingenieuranwendungen...,"Christoph Hübel, Bernd Sutter",book
396,1993,"UNIX für Software-Entwickler - Konzepte, Werkz...","Elmar Buschlinger, Frank Staab",book
397,1993,The Petersen graph.,"Derek A. Holton, John Sheehan",book
398,1993,3D computer graphics (2. ed.).,Alan Watt,book


## 1. RAG implementation


### a. RAG from scratch

Data chunking and segmentation are fundamental techniques in modern data processing, enabling more efficient analysis, retrieval, and generation of information across various media types.

Chunking refers to the process of breaking down larger pieces of data into smaller, more manageable segments that preserve meaningful context while enabling more efficient processing. The primary goal of any chunking strategy is to divide content in a way that maintains coherence and relevance while optimizing for downstream tasks like embedding generation, similarity search, or content analysis.

In [27]:
df

,Year,Title,Author,Type
0,1990,Programming in the 1990s - An Introduction to ...,Edward Cohen,book
1,1990,Programming with Specifications - An Introduct...,David C. Luckham,book
2,1990,First-Order Logic and Automated Theorem Proving,Melvin Fitting,book
3,1990,Specification and Transformation of Programs -...,Helmuth Partsch,book
4,1990,Fundamentals of Algebraic Specification 2,"Hartmut Ehrig, Bernd Mahr",book
...,...,...,...,...
395,1993,Datenbank-Integration von Ingenieuranwendungen...,"Christoph Hübel, Bernd Sutter",book
396,1993,"UNIX für Software-Entwickler - Konzepte, Werkz...","Elmar Buschlinger, Frank Staab",book
397,1993,The Petersen graph.,"Derek A. Holton, John Sheehan",book
398,1993,3D computer graphics (2. ed.).,Alan Watt,book


In [ ]:
# Split data into chunks
# we will use method called "Sementic chunking"

# A. Text segmentation into basic units 
chunks = helper_class.chunking_data(df, chunk_size=1000, overlap=100) # (my implementation)

# B. Embedding



### b. RAG pipeline (LangChain)

Now I will use LangChain. It's easy tool that has important methods for RAG pipeline builded within. Also, I use Opean AI API for LLM - I want to save tokens so I'll use small subdataset (around 400 records).

In [23]:
# API keys for LLM
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

In [ ]:
# Load data from df
loader = DataFrameLoader(df, page_content_column="Title")
docs = loader.load()

# Split ro smaller chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
splits = text_splitter.split_documents(docs)

# Embedding splitted docs
vectorstore = Chroma.from_documents(documents=splits, 
                                    embedding=OpenAIEmbeddings())

# Retrive splitted documents to vector store
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

# Prompt
template = """Jesteś asystentem naukowym. Na podstawie dostarczonego kontekstu wybierz dokładnie 5 artykułów. 
Jeśli w kontekście nie ma wystarczającej liczby artykułów z tego roku, poinformuj o tym użytkownika.
Kontekst: {context}
Pytanie: {question}
Odpowiedź (wypunktowana lista):"""

prompt = ChatPromptTemplate.from_template(template)

# LLM - for bigger context window I use GPT 4o-mini
# temperature 0 to prevent abstraction 
llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)

# Post-processing
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Chain
# pipeline: retrive context (dataframe) from vectorstore, format docs, define question
# use promot from templete, input everything to LLM and parse
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Question from user
rag_chain.invoke("Poleć mi z podanego Pandas DataFrame 5 artykułów, które dotyczą Baz Danych.")

"Na podstawie dostarczonego kontekstu, oto 5 artykułów dotyczących Baz Danych:\n\n1. The Data Mining and Knowledge Discovery Handbook.\n2. The data analysis handbook.\n3. Data Mining - Know It All.\n4. Data warehouse schema design.\n5. Intelligent Access to Data and Knowledge Bases via User's Topics of Interest.\n\nJeśli potrzebujesz więcej informacji na temat tych artykułów, daj znać!"